<a href="https://colab.research.google.com/github/tomasroel/DMeyf_Competencias/blob/main/modelos/1_0_competencia1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LightGBM  Optimizacion Bayesiana

Tarea para el Hogar 04,  versión Gustavo Pasquini: 2026

## 📝 Versión comentada (Tomás)

Este notebook es el original de la cátedra con tres agregados:

* **Rutas**: se adaptaron para trabajar sobre `Mi unidad/competencias_dmeyf/competencia1/` en Drive.
* **Lectura única del dataset**: el crudo se lee una sola vez (en 2.2.4), ahí mismo se calcula `clase_ternaria` y queda en memoria. Ya no se graba ni se relee `competencia_01.csv.gz`.
* **Explicaciones**: las celdas markdown que empiezan con 📝 y los comentarios de código que empiezan con `# >>` son agregados. Los cambios de código están marcados con `# >> CAMBIO`.

El código de modelado (partición, undersampling, BO, entrenamiento final, ganancia) es el original.

**Dónde queda cada cosa**

| Qué | Dónde | ¿Persiste? |
|---|---|---|
| `competencia_01_crudo.csv` | Drive: `competencia1/datasets/` | Sí |
| Copia de trabajo del crudo | Colab: `/content/datasets/` | No |
| `BO_log.txt` | Drive: `competencia1/outputs/<experimento>/` | Sí |

**Cómo correrlo**

1. Runtime en **Python 3**: correr la sección 2.1 (monta Drive y baja el dataset). Es lo único que tiene que ir en Python, porque Colab solo permite montar Drive desde ese runtime.
2. Cambiar el runtime a **R** y correr el resto, de arriba hacia abajo.

Los outputs que se ven guardados en las celdas son de la corrida original (09/09/2026): sirven de referencia para comparar.

## 1.0  Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

📝 **Montaje de Drive.** Conecta la máquina virtual de Colab con tu Google Drive en `/content/.drive`. Sin esto, todo lo que se grabe se pierde al cerrar la sesión.

In [1]:
# primero establecer el Runtime de Python 3
# >> Monta tu Google Drive en /content/.drive (pide autorizacion la primera vez).
# >> "Mi unidad" queda accesible como /content/.drive/My Drive
from google.colab import drive
drive.mount('/content/.drive')

Drive already mounted at /content/.drive; to attempt to forcibly remount, call drive.mount("/content/.drive", force_remount=True).


📝 **Carpetas y descarga del dataset.**

* Crea `competencia1/` en Drive (si no existe) y un link simbólico `/content/buckets/b1` que apunta ahí. A partir de acá, `/content/buckets/b1/...` **es** tu carpeta de Drive.
* Crea `outputs/` y `datasets/` dentro de `competencia1/`, y `/content/datasets` en el disco local.
* `descargar()` baja el crudo del bucket de la materia a Drive solo si no está, y después lo copia al disco local, que es de donde lo lee R (más rápido que leer de Drive).

In [2]:
%%shell

# >> CAMBIO: carpeta base en Drive = Mi unidad/competencias_dmeyf/competencia1
# >> (original: "/content/.drive/My Drive/dmeyf")
mkdir -p "/content/.drive/My Drive/competencias_dmeyf/competencia1"
mkdir -p "/content/buckets"
# >> link simbolico: /content/buckets/b1 apunta a la carpeta de Drive,
# >> asi el codigo R usa una ruta corta y sin espacios
ln -sfn  "/content/.drive/My Drive/competencias_dmeyf/competencia1" /content/buckets/b1


# >> CAMBIO: los resultados van a competencia1/outputs (original: b1/exp)
mkdir -p /content/buckets/b1/outputs
# >> competencia1/datasets en Drive: copia persistente del dataset crudo
mkdir -p /content/buckets/b1/datasets
# >> /content/datasets: disco local de Colab (rapido, pero se borra al cerrar la sesion)
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/dmeyf2026-9c6f/"
  archivo="$1"

  # >> paso 1: si el archivo NO esta en Drive, lo baja del bucket de la materia
  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  # >> paso 2: si NO esta en el disco local, lo copia desde Drive
  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "competencia_01_crudo.csv"


## 2.0 Train y Optimizacion Hiperparámetros

Esta parte se debe correr con el runtime en lenguaje R Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

### 2.1 Inicio

limpio el ambiente de R

In [1]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sun Oct 04 11:19:28 PM 2026"

📝 Limpia la memoria de R antes de empezar. El dataset todavía no está cargado: se lee más abajo, en 2.2.4.

In [2]:
# limpio la memoria
# >> borra todos los objetos que haya en memoria antes de empezar
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671297,35.9,1473300,78.7,1473300,78.7
Vcells,1242690,9.5,8388608,64.0,1978689,15.1


### 2.2 Carga de Librerias

Esta parte lleva  demasiados   minutos la primera vez en Google Colab
<br> por suerte a partir del 15-septiembre empezaremos a trabajar con Google Cloud, en donde todas las librerias, de Python R y Julia ya estarán instaladas

In [3]:
# instalo paquetes
# >> R.utils: utilidades (lectura de .gz) / rlist: manejo de listas (list.names en loguear)
install.packages( c("R.utils", "rlist" ) )

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘R.oo’, ‘R.methodsS3’, ‘XML’




In [4]:
# Librerias  modernas  para la Bayesian Optimization
# >> bbotk: framework de optimizacion black-box / mlr3mbo: optimizacion bayesiana
# >> DiceKriging: el Gaussian Process (kriging) que usa la BO como modelo surrogate
install.packages( c("bbotk", "mlr3", "mlr3mbo", "mlr3learners", "DiceKriging") )

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘rbibutils’, ‘matrixStats’, ‘Rdpack’, ‘globals’, ‘listenv’, ‘nanonext’, ‘PRROC’, ‘paradox’, ‘checkmate’, ‘lgr’, ‘mlr3misc’, ‘moocore’, ‘future’, ‘future.apply’, ‘mirai’, ‘mlbench’, ‘mlr3measures’, ‘parallelly’, ‘palmerpenguins’, ‘mlr3tuning’, ‘spacefillr’




In [5]:
# el LightGBM
install.packages(c("lightgbm"))

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [6]:
library(data.table)
library(R.utils)
library(rlist)
library(bbotk)
library(mlr3)
library(mlr3mbo)
library(mlr3learners)
# library(paradox)
library(DiceKriging)
library(lightgbm)


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Attaching package: ‘R.utils’


The following object is masked from ‘package:utils’:

    timestamp


The following objects are masked from ‘package:base’:

    cat, commandArgs, getOption, isOpen, nullfile, parse, use, warnings


Loading required package: paradox


Attaching package: ‘ml

### 2.3 Definicion de Parametros

aqui debe cargar SU semilla primigenia
<br>recuerde cambiar el numero de experimento en cada corrida nueva

📝 **Parámetros del experimento.** Todo lo configurable vive en la lista `PARAM`. `PARAM$experimento` define el nombre de la subcarpeta de salida en `competencia1/outputs/`.

In [9]:
PARAM <- list()
# >> nombre del experimento = nombre de la subcarpeta en competencia1/outputs/
# >> cambiarlo en cada corrida nueva (si no, el BO_log.txt se acumula sobre el anterior)
PARAM$experimento <- "base_010"
# >> semilla usada para la particion, el undersampling y LightGBM (se deja la original)
PARAM$semilla_primigenia <-  911341   # La semilla de Laura Pasquini


📝 **Esquema temporal.** La BO y el modelo final usan 202104 (último mes con clase completa si estamos "parados" en 202106). El modelo se aplica sobre 202106 y, como ahí la clase es conocida, se puede medir la ganancia real para distintos cortes de envíos.

In [10]:
# training y future
# >> 70% de 202104 a training, 30% a validation
PARAM$training_pct <- 70
# >> mes usado en la optimizacion bayesiana
PARAM$train <- c(202104)
# >> mes usado para entrenar el modelo final
PARAM$train_final <- c(202104)
# >> mes donde se aplica el modelo y se mide la ganancia (tiene clase conocida)
PARAM$future <- c(202106)
# >> cantidades de envios a evaluar: 9000, 9500, ..., 15000
PARAM$cortes <- seq(9000, 15000, by= 500)

📝 **Undersampling.** Para entrenar se usan todos los BAJA y solo el 10% de los CONTINUA. Acelera mucho el entrenamiento, a costa de que las probabilidades que predice el modelo dejan de estar en la escala real.

In [11]:
# un undersampling de 0.1  toma solo el 10% de los CONTINUA
# undersampling de 1.0  implica tomar TODOS los datos

PARAM$trainingstrategy$undersampling <- 0.1

📝 **Iteraciones de la BO.** Con 3 hiperparámetros, `mlr3mbo` arranca con un diseño inicial de 4 × 3 = 12 puntos que se evalúan todos juntos. Como el tope es 10, la optimización termina apenas se evalúa ese diseño inicial: en la corrida original el log muestra `Evaluating 12 configuration(s)` y `Finished optimizing after 12 evaluation(s)`. Es decir, con este valor se prueban 12 puntos de exploración inicial y ninguno llega a ser propuesto por el modelo surrogate.

In [12]:
# >> tope de evaluaciones de la BO (ver nota en la celda markdown de arriba)
PARAM$hyperparametertuning$iteraciones <- 20 # iteraciones bayesianas

📝 **Parámetros fijos de LightGBM.** Son los que no se optimizan. Lo central de la estrategia:

* `learning_rate = 0.005` muy bajo y fijo, compensado con muchas iteraciones.
* `feature_fraction = 0.5` y `max_bin = 31` como regularización.
* `min_data_in_leaf = 0`: el tamaño mínimo de hoja se controla solo con `min_sum_hessian_in_leaf`.
* Sin early stopping y sin rebalanceo interno (`is_unbalance = FALSE`, `scale_pos_weight = 1`).
* Los tres últimos (`num_iterations`, `num_leaves`, `min_sum_hessian_in_leaf`) son los que pisa la BO.

In [13]:
# Parametros LightGBM

# parametros fijos del LightGBM que se pisaran con la parte variable de la BO
PARAM$lgbm$param_fijos <-  list(
  # >> gradient boosting clasico
  boosting= "gbdt", # puede ir  dart  , ni pruebe random_forest
  # >> clasificacion binaria (clase01), minimiza logloss
  objective= "binary",
  # >> metrica que se calcula en validation: area bajo la curva precision-recall
  metric= "average_precision",
  feature_pre_filter= FALSE,
  first_metric_only= FALSE,
  boost_from_average= TRUE,
  force_row_wise= TRUE,
  deterministic= TRUE,
  verbosity= -100,

  # >> semilla interna de LightGBM
  seed= PARAM$semilla_primigenia,

  # >> sin limite de profundidad; la complejidad se controla con num_leaves
  max_depth= -1L,
  # >> sin ganancia minima para abrir un split y sin regularizacion L1 ni L2
  min_gain_to_split= 0,
  lambda_l1= 0.0,
  lambda_l2= 0.0,
  # >> cada variable numerica se discretiza en 31 bins (mas rapido, regulariza)
  max_bin= 31L,

  # >> sin bagging de filas: cada arbol ve todas las filas del training
  bagging_fraction= 1.0,
  pos_bagging_fraction= 1.0,
  neg_bagging_fraction= 1.0,
  # >> NO se rebalancea dentro de LightGBM (el desbalance se trata con el undersampling)
  is_unbalance= FALSE,
  scale_pos_weight= 1.0,

  # >> drop_rate / max_drop / skip_drop solo aplican a boosting=dart; aca no tienen efecto
  drop_rate= 0.1,
  max_drop= 50,
  skip_drop= 0.5,

  extra_trees= FALSE,

  # >> 0 = sin early stopping: siempre se entrenan las num_iterations completas
  early_stopping= 0,
  # >> sin minimo de registros por hoja; el tamaño minimo de hoja se regula
  # >> con min_sum_hessian_in_leaf, que es el que optimiza la BO
  min_data_in_leaf= 0,     # fijo, esto es FUNDAMENTAL
  # >> cada arbol usa el 50% de las variables, elegidas al azar
  feature_fraction= 0.50,  # fijo
  # >> learning rate muy bajo y fijo; se compensa con muchas iteraciones
  learning_rate= 0.005,    # fijo

  # >> los tres de abajo son placeholders: los pisa cada iteracion de la BO
  num_iterations= 20000,     # se modificara
  num_leaves= 1024,          # se modificara
  min_sum_hessian_in_leaf= 1e-06  # se modificara
)


Aqui se definen los hiperparámetros de LightGBM que participan de la Bayesian Optimization

📝 **Espacio de búsqueda.** Rangos de los 3 hiperparámetros que explora la BO. `ps()`, `p_int()` y `p_dbl()` son de la librería `paradox`.

In [14]:
# Aqui se cargan los bordes de los hiperparametros de la BO
#  es vital entender que se optimiza min_sum_hessian_in_leaf y que  min_data_in_leaf se deja fijo en CERO

PARAM$hypeparametertuning$hs <- ps(
  # >> cantidad de arboles: entero entre 64 y 4096
  num_iterations= p_int(lower= 64L,  upper= 4096L),
  # >> hojas maximas por arbol: entero entre 16 y 2048
  num_leaves= p_int(lower= 16L,  upper= 2048L),
  # >> suma minima de hessianos por hoja: real entre 1e-06 y 0.1
  # >> (en logloss el hessiano de una fila es p*(1-p): funciona como un 'conteo blando' de registros)
  min_sum_hessian_in_leaf= p_dbl( lower=1e-06, upper=0.1)
)

📝 **Función `particionar()`.** Agrega al dataset una columna `fold` con una partición al azar y estratificada (acá 70/30 por `clase_ternaria`).

In [15]:
# particionar agrega una columna llamada fold a un dataset
#   que consiste en una particion estratificada segun agrupa
# particionar( data=dataset, division=c(70,30),
#  agrupa=clase_ternaria, seed=semilla)   crea una particion 70, 30

particionar <- function(data, division, agrupa= "", campo= "fold", start= 1, seed= NA) {
  # >> fija la semilla para que la particion sea reproducible
  if (!is.na(seed)) set.seed(seed, "L'Ecuyer-CMRG")

  # >> arma un bloque con 70 unos y 30 doses (segun `division`)
  bloque <- unlist(mapply(
    function(x, y) {rep(y, x)},division, seq(from= start, length.out= length(division))))

  # >> dentro de cada clase (by= agrupa) repite el bloque, lo mezcla y asigna fold 1 o 2:
  # >> particion estratificada, cada clase queda 70/30
  data[, (campo) := sample(rep(bloque,ceiling(.N / length(bloque))))[1:.N],by= agrupa]
}

📝 **Función `loguear()`.** Agrega una fila a un archivo de texto separado por tabs, con fecha-hora y todos los valores de la lista que recibe. Así se arma `BO_log.txt`, una fila por cada modelo evaluado en la BO.

In [16]:
# logueo al archivo BO_log.txt
loguear  <- function( preg, arch=NA, verbose=TRUE )
{
  t0 <- Sys.time()
  reg <- copy( preg )
  # >> (no aplica en este notebook: param_fijos no tiene feature_contri)
  if( "feature_contri" %in%  names(reg) ) {
    reg$feature_contri <- reg$feature_contri[1]
  }
  archivo <- arch
  if( is.na(arch) ) archivo <- paste0( folder, substitute( reg), ext )


  # >> si el archivo no existe, primero escribe la fila de titulos
  if( !file.exists( archivo ) )
  {
    # Escribo los titulos
    linea  <- paste0( "fecha\t",
                      paste( list.names(reg), collapse="\t" ), "\n" )

    cat( linea, file=archivo )
  }

  # escribo el registro
  # >> una fila por evaluacion: fecha-hora + todos los valores, separados por tab
  linea  <- paste0( format(t0, "%Y%m%d.%H%M%S"),  "\t",     # la fecha y hora
                    gsub( ", ", "\t", toString( reg ) ),  "\n" )

  cat( linea, file=archivo, append=TRUE )  # grabo al archivo

  if( verbose )  cat( linea )   # imprimo por pantalla
}

### 2.4  Preprocesamiento

📝 **Carpeta de trabajo.** Se para en `competencia1/outputs/<experimento>/` (en Drive). Todo archivo que se grabe con ruta relativa, como `BO_log.txt`, queda ahí.

In [ ]:
# carpeta de trabajo

# >> CAMBIO: carpeta de salida en Drive = competencia1/outputs (original: "/content/buckets/b1/exp")
setwd("/content/buckets/b1/outputs")
# >> crea la subcarpeta del experimento (outputs/HT4950-01) y se para ahi
dir.create(PARAM$experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/outputs/", PARAM$experimento))

📝 **Lectura del dataset y cálculo de `clase_ternaria` (CAMBIO: una sola lectura).** En el original el crudo se leía al principio, se le calculaba la clase, se grababa como `competencia_01.csv.gz` y acá se volvía a leer. Ahora se lee una sola vez en este punto, se calcula la clase y `dataset` queda en memoria para el resto del notebook.

El crudo no trae la clase: se deduce de si el cliente aparece o no en los meses siguientes.

* **BAJA+1**: no está en la foto del mes siguiente.
* **BAJA+2**: está en el mes siguiente, pero no en el subsiguiente.
* **CONTINUA**: está en los dos meses siguientes.

Como hacen falta dos meses de futuro, la clase solo queda completa hasta 202106. En 202107 solo se pueden marcar los BAJA+1 (el resto queda `NA`) y 202108 queda todo en `NA`.

Diferencias con el código original de esta celda: se sacó `require("data.table")` (ya se cargó en 2.2.2), se agregó `stringsAsFactors= TRUE` al `fread`, se sacó el `fwrite` y se agregó `rm(dsimple)`. El cálculo de la clase no cambia.

In [ ]:
# leo el dataset
# >> CAMBIO: unica lectura del dataset en todo el notebook (crudo, sin clase_ternaria)
# >> se agrega stringsAsFactors= TRUE, que era la opcion de la lectura que usaba el modelo
dataset <- fread("/content/datasets/competencia_01_crudo.csv", stringsAsFactors= TRUE )

# calculo el periodo0 consecutivo
# >> tabla auxiliar minima: posicion original de la fila, cliente y periodo0
# >> periodo0 = año*12 + mes -> numera los meses de corrido (202112 y 202201 quedan consecutivos)
dsimple <- dataset[, list(
    "pos" = .I,
    numero_de_cliente,
    periodo0 = as.integer(foto_mes/100)*12 +  foto_mes%%100 ) ]


# ordeno
setorder( dsimple, numero_de_cliente, periodo0 )

# calculo topes
# >> ultimo mes del dataset (202108) y anteultimo (202107)
periodo_ultimo <- dsimple[, max(periodo0) ]
periodo_anteultimo <- periodo_ultimo - 1


# calculo los leads de orden 1 y 2
# >> para cada cliente: periodo1 = el proximo mes en que aparece, periodo2 = el siguiente
# >> (NA si el cliente no vuelve a aparecer)
dsimple[, c("periodo1", "periodo2") :=
    shift(periodo0, n=1:2, fill=NA, type="lead"),  numero_de_cliente ]

# assign most common class values = "CONTINUA"
# >> por defecto CONTINUA, solo para meses con 2 meses de futuro observable (hasta 202106)
dsimple[ periodo0 < periodo_anteultimo, clase_ternaria := "CONTINUA" ]

# calculo BAJA+1
# >> BAJA+1: el cliente NO esta en el mes siguiente (no hay registro o hay un hueco)
# >> se puede calcular hasta 202107 (alcanza con 1 mes de futuro)
dsimple[ periodo0 < periodo_ultimo &
    ( is.na(periodo1) | periodo0 + 1 < periodo1 ),
    clase_ternaria := "BAJA+1" ]

# calculo BAJA+2
# >> BAJA+2: esta en el mes siguiente pero NO en el subsiguiente
# >> se puede calcular hasta 202106; 202107 queda con BAJA+1 o NA y 202108 todo NA
dsimple[ periodo0 < periodo_anteultimo & (periodo0+1 == periodo1 )
    & ( is.na(periodo2) | periodo0 + 2 < periodo2 ),
    clase_ternaria := "BAJA+2" ]


# pego el resultado en el dataset original
# >> vuelve al orden original de filas para pegar la clase en el dataset completo
setorder( dsimple, pos )
dataset[, clase_ternaria := dsimple$clase_ternaria ]

# >> CAMBIO: ya no se graba ni se relee competencia_01.csv.gz;
# >> `dataset` queda en memoria, con clase_ternaria, para el resto del notebook
# >> libero la tabla auxiliar
rm( dsimple )

📝 **Control.** Tabla de cantidad de clientes por mes y clase. En la corrida original, 202104 tiene 1.139 BAJA+2, 964 BAJA+1 y 161.181 CONTINUA; 202106 tiene 1.098 BAJA+2.

CAMBIO: el original hacía `setorder(dataset, ...)`, que reordena las filas de `dataset`. Antes no importaba, porque el dataset se volvía a leer desde el archivo. Ahora `dataset` es el que se usa para modelar, y el orden de filas define la partición 70/30 y el undersampling, así que se cuenta con `keyby` sin tocar el orden.

In [ ]:
# >> CAMBIO: keyby ordena solo la tabla de conteos y NO reordena `dataset`
# >> (original: setorder( dataset, foto_mes, clase_ternaria, numero_de_cliente) )
dataset[, .N, keyby= list(foto_mes, clase_ternaria)]

foto_mes,clase_ternaria,N
<int>,<chr>,<int>
202103,BAJA+1,1019
202103,BAJA+2,960
202103,CONTINUA,160921
202104,BAJA+1,964
202104,BAJA+2,1139
202104,CONTINUA,161181
202105,BAJA+1,1143
202105,BAJA+2,870
202105,CONTINUA,161755


📝 **Armado de training y validation** (esta celda y las cuatro siguientes):

1. Se queda con 202104.
2. Particiona 70/30 estratificado (`fold` 1 y 2).
3. Crea `clase01`: BAJA+1 y BAJA+2 valen 1, CONTINUA vale 0.
4. Marca `training = 1` para el fold 1 con undersampling (todos los BAJA + 10% de los CONTINUA).
5. Define las variables predictoras (`campos_buenos`).

En la corrida original quedan 12.778 filas en training y 150.506 en validation, con 152 variables.

Tener en cuenta que validation es todo lo que tiene `training = 0`: el fold 2 completo **más** los CONTINUA del fold 1 que no salieron sorteados. No hay filas compartidas con training, pero la proporción de BAJA en validation es menor que la real del mes.

In [ ]:
# >> se queda solo con 202104
dataset_train <- dataset[foto_mes %in% PARAM$train]

In [ ]:
  # >> agrega la columna fold: 1 = 70%, 2 = 30%, estratificado por clase_ternaria
  particionar(dataset_train,
    division= c(PARAM$training_pct, 100L -PARAM$training_pct),
    agrupa= "clase_ternaria",
    seed= PARAM$semilla_primigenia # aqui se usa SU semilla
  )

In [ ]:
# paso la clase a binaria que tome valores {0,1}  enteros
#  BAJA+1 y BAJA+2  son  1,   CONTINUA es 0
#  a partir de ahora ya NO puedo cortar  por prob(BAJA+2) > 1/40

# >> al juntar BAJA+1 con BAJA+2 la probabilidad que predice el modelo es P(BAJA+1 o BAJA+2),
# >> por eso ya no vale el corte teorico 1/40 y despues se corta por cantidad de envios
dataset_train[,
  clase01 := ifelse(clase_ternaria %in% c("BAJA+2","BAJA+1"), 1L, 0L)
]

In [ ]:
# defino los datos que forma parte del training
# aqui se hace el undersampling de los CONTINUA
# notar que para esto utilizo la SEGUNDA semilla

# >> ojo: el comentario original dice SEGUNDA semilla, pero el codigo usa la primigenia
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
# >> numero al azar uniforme [0,1] por fila
dataset_train[, azar := runif(nrow(dataset_train))]
dataset_train[, training := 0L]

# >> training = 1 solo para filas del fold 1 que sean BAJA (todas) o CONTINUA con azar <= 0.1
# >> todo lo demas queda en training = 0 (fold 2 completo + los CONTINUA del fold 1 no sorteados)
dataset_train[
  foto_mes %in%  PARAM$train &  fold==1 &
    (azar <= PARAM$trainingstrategy$undersampling | clase_ternaria %in% c("BAJA+1", "BAJA+2")),
  training := 1L
]

In [ ]:
# los campos que se van a utilizar

# >> variables predictoras = todas las columnas menos la clase y las auxiliares
# >> (numero_de_cliente y foto_mes SI quedan como predictoras)
campos_buenos <- setdiff(
  colnames(dataset_train),
  c("clase_ternaria", "clase01", "fold", "azar", "training")
)

#### 2.4.1  Salsa Magica (dedicada a Laura Pasquini)

Los EDAs que estan haciendo los alumnos de los lunes son pura falopa, y de la mala. El uso de Agentes de la IA solo les empera el daño cerebral, generan pedf larguisimos, que no te dan ganas de leerlos, carentes de insights y alma.
<br>El unico EDA que tiene realmente sentido, en este caso, es comparar la distribución de cada variable del mes donde se entrena 202104 y el mes donde se va a aplicar el modelo 202106, buscando Data Drifting.
<br>Aun con los reales datos del futuro, donde no se tiene la clase, SIEMPRE se puede analizar el Data Drifting.
<br>Para las variables con distribuciones distintas se puede hacer:
* Transformar la variable para hacerla homogenea entre los dos periodos
* Eliminarla (es lo simple, que se eligió hacer en este caso)

En esa exploracion gráfica del Data Drifting, se vio que estas variables cambian de distribucion entre 202104 y 202106, y se decidió brutalmente eliminarlas
* cprestamos_personales
* mprestamos_personales


más adelantes, se utilizarán técnicas más inclusivas, y se preservaran dichas variables, transformadas.


In [ ]:
# >> saca las dos variables con data drifting entre 202104 y 202106
campos_buenos <- setdiff( campos_buenos, c("cprestamos_personales", "mprestamos_personales") )

2.2.5 Configuracion Bayesian Optimization

📝 **Contador de iteraciones.** `giter` numera las evaluaciones en el log. Si en la carpeta ya existe un `BO_log.txt` (corrida anterior con el mismo nombre de experimento), continúa la numeración y las filas nuevas se agregan al mismo archivo.

In [ ]:
# >> contador global de evaluaciones de la BO
giter <- 0

# >> si ya hay un BO_log.txt en la carpeta del experimento, sigue numerando desde ahi
# >> (solo continua la numeracion del log; la BO en si arranca de cero)
if( file.exists("BO_log.txt") ){
  tb_BO <- fread("BO_log.txt")
  giter <- tb_BO[, max(iter)]
}


📝 **Función objetivo de la BO.** Por cada combinación de hiperparámetros: entrena un LightGBM en training, mide `average_precision` en validation, graba la fila en `BO_log.txt` y devuelve la métrica.

La BO maximiza **average precision**, no ganancia (el comentario original "obtengo la ganancia" no refleja lo que hace el código).

In [ ]:
# En el argumento x llegan los parametros de la bayesiana

Estimar_lightgbm <- function(xs) {

  flush.console()
  giter <<- giter + 1

  # x pisa (o agrega) a param_fijos
  # >> combina los parametros fijos con los 3 que propone la BO
  param_completo <- modifyList(PARAM$lgbm$param_fijos, xs)
  xx <- copy(param_completo)

  # entreno LightGBM
  # >> entrena en dtrain y evalua average_precision en dvalidate en cada iteracion
  modelo <- lgb.train(
    data= dtrain,
    valids= list( "val"=dvalidate),
    param= param_completo
  )


  # obtengo la ganancia
  xx$iter <- giter
  # >> ojo: aca NO se calcula ganancia. best_score = mejor average_precision en validation
  # >> a lo largo de las iteraciones del boosting
  xx$metrica <- modelo$best_score

  # >> agrega una fila a BO_log.txt (queda en Drive, en la carpeta del experimento)
  loguear( xx, "BO_log.txt")
  set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")  # le reordeno a mlr3MBO

  # >> valor que la BO va a maximizar
  return( list( metrica = xx$metrica) )
}

📝 **Datasets en formato LightGBM.** Convierte training y validation a `lgb.Dataset` (matriz numérica + etiqueta `clase01`).

In [ ]:
# dejo los datos de training en el formato que necesita LightGBM

# >> training: filas con training == 1 (fold 1 con undersampling), solo campos_buenos
dtrain <- lgb.Dataset(
  data= data.matrix(dataset_train[training == 1L, campos_buenos, with= FALSE]),
  label= dataset_train[training == 1L, clase01],
  free_raw_data= FALSE
)

nrow(dtrain)
ncol(dtrain)

[1] 12778

[1] 152

In [ ]:
# datos de validatin en el formato que necesita LightGBM
# >> validation: TODAS las filas con training == 0, sin undersampling
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset_train[training == 0L, campos_buenos, with= FALSE]),
  label= dataset_train[training == 0L, clase01],
  free_raw_data= FALSE
)

nrow(dvalidate)
ncol(dvalidate)

[1] 150506

[1] 152

📝 **Configuración de la BO** con `bbotk` + `mlr3mbo`:

* **Objetivo**: `Estimar_lightgbm`, sobre el espacio de búsqueda definido antes, maximizando `metrica`.
* **Terminador**: corta al alcanzar `PARAM$hyperparametertuning$iteraciones` evaluaciones.
* **Surrogate**: Gaussian Process (`regr.km`, de DiceKriging).
* **Adquisición**: Expected Improvement, optimizada por búsqueda aleatoria de 1.000 puntos.
* **Loop**: `bayesopt_ego` (Efficient Global Optimization).

In [ ]:
# Aqui comienza la configuracion de la Bayesian Optimization

# en este archivo quedan la evolucion binaria de la BO
# >> (esta variable no se usa mas abajo)
kbayesiana <- "bayesiana.RDATA"

funcion_optimizar <- Estimar_lightgbm # la funcion que voy a maximizar

# >> funcion objetivo: recibe los hiperparametros (domain) y devuelve `metrica` a maximizar
objective <- ObjectiveRFun$new(
  fun= funcion_optimizar,
  domain= PARAM$hypeparametertuning$hs,
  codomain= ps(metrica = p_dbl(tags="maximize")),
  id= "BO_lightgbm"
)


# >> instancia de optimizacion: objetivo + criterio de corte por cantidad de evaluaciones
instance <- OptimInstanceBatchSingleCrit$new(
  objective= objective,
  terminator=  trm("evals", n_evals = PARAM$hyperparametertuning$iteraciones)
)


# >> surrogate: Gaussian Process (kriging) que aproxima metrica = f(hiperparametros)
# >> nugget.estim=TRUE: asume que la metrica observada tiene ruido
surrogate <- srlrn(lrn("regr.km",
  optim.method = "BFGS",
  nugget.estim = TRUE,
  control= list(trace = FALSE)))

# >> funcion de adquisicion: elige el proximo punto por Expected Improvement
acq_function  <- acqf("ei")                 # expected improvement

# >> el maximo de EI se busca probando 1000 puntos al azar sobre el surrogate
acq_optimizer  <- acqo(
  optimizer  = opt("random_search", batch_size= 1000),
  terminator = trm("evals", n_evals= 1000)
)

# >> arma el optimizador bayesiano (EGO) con las tres piezas anteriores
optimizer <- opt("mbo",
  loop_function = bayesopt_ego,
  surrogate= surrogate,
  acq_function= acq_function,
  acq_optimizer= acq_optimizer
)



2.2.6 Corrida Bayesian Optimization

📝 **Corrida.** Ejecuta la optimización. Cada evaluación imprime su fila del log. En la corrida original el mejor punto fue `num_iterations = 2848`, `num_leaves = 71`, `min_sum_hessian_in_leaf = 0.0067`, con average precision 0,1230.

In [ ]:
# >> corre la BO: es la celda mas lenta del notebook (~45 min en la corrida original)
bayesiana_salida <- optimizer$optimize(instance)

INFO  [19:00:45.280] [bbotk] Starting to optimize 3 parameter(s) with '<OptimizerMbo>' and '<TerminatorEvals> [n_evals=10, k=0]'
INFO  [19:00:45.453] [bbotk] Evaluating 12 configuration(s)
20260909.190416	gbdt	binary	average_precision	FALSE	FALSE	TRUE	TRUE	TRUE	-100	585581	-1	0	0	0	31	1	1	1	FALSE	1	0.1	50	0.5	FALSE	0	0	0.5	0.005	3565	76	0.0121267294716439	1	0.122165219183793
20260909.191102	gbdt	binary	average_precision	FALSE	FALSE	TRUE	TRUE	TRUE	-100	585581	-1	0	0	0	31	1	1	1	FALSE	1	0.1	50	0.5	FALSE	0	0	0.5	0.005	3632	1514	0.0132329566883098	2	0.107730070825482
20260909.191544	gbdt	binary	average_precision	FALSE	FALSE	TRUE	TRUE	TRUE	-100	585581	-1	0	0	0	31	1	1	1	FALSE	1	0.1	50	0.5	FALSE	0	0	0.5	0.005	2763	1180	0.076964120381018	3	0.111865098739407
20260909.191807	gbdt	binary	average_precision	FALSE	FALSE	TRUE	TRUE	TRUE	-100	585581	-1	0	0	0	31	1	1	1	FALSE	1	0.1	50	0.5	FALSE	0	0	0.5	0.005	970	1106	0.088472420655413	4	0.100073146587283
20260909.192023	gbdt	binary	average_precision	FALSE	

Luego de este punto, en la carpeta del experimento quedó el archivo BO_log.txt que posee todas las iteraciones de la Bayesian Optimization

## 3.0  Produccion

### 3.1 Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

In [ ]:
# >> CAMBIO: misma carpeta de salida en Drive (original: "/content/buckets/b1/exp")
setwd("/content/buckets/b1/outputs")
dir.create(PARAM$experimento, showWarnings= FALSE)
setwd( paste0("/content/buckets/b1/outputs/", PARAM$experimento ))

📝 **Mejores hiperparámetros.** Lee `BO_log.txt` de la carpeta del experimento, lo ordena por métrica descendente y toma la primera fila. Si el log acumula corridas anteriores, las considera a todas.

In [ ]:
 # cergo los mejores hiperparametros, que son los de mayor metrica
 # >> lee el log de la BO y se queda con la fila de mayor metrica
 tb_bayesiana <- fread("BO_log.txt")
 setorder( tb_bayesiana, -metrica )
 PARAM$out$lgbm$mejores_hiperparametros <- as.list(tb_bayesiana[1])

#### 3.2 Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la  optimización bayesiana

📝 **Dataset de entrenamiento final.** A diferencia de la BO, acá se entrena con **todo** 202104: sin partición 70/30 y sin undersampling (163.284 filas contra 12.778). Los hiperparámetros se eligieron sobre un dataset mucho más chico que el que se usa en el modelo final.

In [ ]:
# clase01
# >> clase binaria ahora sobre el dataset completo (todos los meses)
dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+1", "BAJA+2"), 1L, 0L)]

In [ ]:
# >> pisa dataset_train: ahora es TODO 202104, sin particion y sin undersampling
dataset_train <- dataset[foto_mes %in% PARAM$train_final]
dataset_train[,.N, list(foto_mes, clase_ternaria)]

foto_mes,clase_ternaria,N
<int>,<fct>,<int>
202104,CONTINUA,161181
202104,BAJA+2,1139
202104,BAJA+1,964


#### 3.3 Final Training Hyperparameters

In [ ]:
# >> parametros fijos + mejores hiperparametros de la BO
# >> (tambien entran fecha, iter y metrica del log, que no son parametros de LightGBM)
param_final <- modifyList(PARAM$lgbm$param_fijos,
  PARAM$out$lgbm$mejores_hiperparametros)

param_final

$boosting
[1] "gbdt"

$objective
[1] "binary"

$metric
[1] "average_precision"

$feature_pre_filter
[1] FALSE

$first_metric_only
[1] FALSE

$boost_from_average
[1] TRUE

$force_row_wise
[1] TRUE

$deterministic
[1] TRUE

$verbosity
[1] -100

$seed
[1] 585581

$max_depth
[1] -1

$min_gain_to_split
[1] 0

$lambda_l1
[1] 0

$lambda_l2
[1] 0

$max_bin
[1] 31

$bagging_fraction
[1] 1

$pos_bagging_fraction
[1] 1

$neg_bagging_fraction
[1] 1

$is_unbalance
[1] FALSE

$scale_pos_weight
[1] 1

$drop_rate
[1] 0.1

$max_drop
[1] 50

$skip_drop
[1] 0.5

$extra_trees
[1] FALSE

$early_stopping
[1] 0

$min_data_in_leaf
[1] 0

$feature_fraction
[1] 0.5

$learning_rate
[1] 0.005

$num_iterations
[1] 2848

$num_leaves
[1] 71

$min_sum_hessian_in_leaf
[1] 0.006708277

$fecha
[1] 20260909

$iter
[1] 8

$metrica
[1] 0.1230107

In [ ]:
# aplico el modelo a los datos sin clase
# >> 202106: mes de testeo. Tiene clase conocida, por eso se puede medir ganancia
dfuture <- dataset[foto_mes %in% PARAM$future]


📝 **Entrenamiento final y predicción.** Entrena un único LightGBM con los mejores hiperparámetros y predice la probabilidad de baja para cada cliente de 202106.

In [ ]:
# Entreno el modelo sobre todo el periodo

dtrain_final <- lgb.Dataset(
  data= data.matrix(dataset_train[, campos_buenos, with= FALSE]),
  label= dataset_train[, clase01]
)

# >> un unico modelo, sin validation: entrena las num_iterations elegidas
modelo_final <- lgb.train(
  data= dtrain_final,
  param= param_final
)

In [ ]:
# hago el predict en los datos del futuro, y calculo la ganancia

# aplico el modelo a los datos nuevos
# >> probabilidad de clase01 = 1 para cada cliente de 202106
prediccion <- predict(
  modelo_final,
  data.matrix(dfuture[, campos_buenos, with= FALSE])
)



📝 **Ganancia.** Ordena los clientes de 202106 por probabilidad descendente y, para cada corte de `PARAM$cortes`, suma la ganancia de enviar la campaña a los primeros N. Cada BAJA+2 acertado suma 1.072.500 y cualquier otro envío resta 27.500 (los BAJA+1 se entrenaron como positivos, pero acá restan).

En la corrida original el máximo es **431.750.000 con 11.500 envíos**.

La tabla solo se imprime en pantalla: no se graba en Drive. Tampoco se guarda el modelo (`lgb.save` está comentado). Lo único que persiste del experimento es `BO_log.txt`.

In [ ]:
# calculo la ganancia en los datos ddel futuro para todos los cortes
tb_prediccion <- dfuture[, list(numero_de_cliente, foto_mes,clase_ternaria)]
tb_prediccion[, prob := prediccion ]
# >> ganancia por envio: acierto BAJA+2 = 2.200.000 * 50% - 27.500 = 1.072.500
# >> cualquier otro (CONTINUA o BAJA+1) = -27.500
tb_prediccion[, gan := ifelse(clase_ternaria == "BAJA+2", 1072500, -27500)]

# lgb.save(modelo_final, "modelo.txt" )
# Dibujo la curva de ganancia acumulada
# >> ordena de mayor a menor probabilidad y acumula la ganancia
setorder(tb_prediccion, -prob)
tb_prediccion[, ganancia_acumulada := cumsum(gan)]
tb_prediccion[, pos := sequence(.N)]

# >> ganancia de enviar la campaña a los primeros `envios` clientes del ranking
# >> solo se imprime en pantalla: no se graba ningun archivo
for (envios in PARAM$cortes) {
  ganancia <- tb_prediccion[1:envios, sum(gan)]
  cat( envios, "\t", ganancia, "\n")
}

9000 	 408100000 
9500 	 409750000 
10000 	 416900000 
10500 	 419650000 
11000 	 427900000 
11500 	 431750000 
12000 	 4.29e+08 
12500 	 430650000 
13000 	 4.29e+08 
13500 	 424050000 
14000 	 422400000 
14500 	 424050000 
15000 	 420200000 


In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Wed Sep 09 07:53:25 PM 2026"